# Hindawi 2026.10 in Pashto

Program in Pashto: every GATE program, in C and in Python, compiled through the original Hindawi pipeline, with every step shown.

© 1993–2026 Abhishek Choudhary. All rights reserved. AyeAI. Hindawi™ and Romenagri™ are his trademarks; AyeAI®, AyeCNSe® and AyeAM® are his registered trademarks in India.

## 1. Installation

Run this once. It fetches Hindawi, builds the Romenagri tools and the original transducers, and loads the compilation path.

In [ ]:
LANG = 'pashto'
SECTIONS = ['algorithms/analysis', 'algorithms/backtracking', 'algorithms/divide_and_conquer', 'algorithms/dynamic_programming', 'algorithms/graphs', 'algorithms/greedy', 'algorithms/searching', 'algorithms/sorting', 'compiler_design', 'computer_networks', 'computer_organization', 'data_structures', 'databases', 'digital_logic', 'discrete_mathematics', 'engineering_mathematics', 'operating_systems', 'paradigms', 'programming', 'theory_of_computation']
import os, subprocess, sys, tempfile, pathlib
REPO = os.environ.get('HINDAWI_REPO') or os.path.abspath('hindawi2026.10')
if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone', '-q', '--depth', '1', 'https://github.com/hindawiai/hindawi2026.10', REPO], check=True)
if 'google.colab' in sys.modules:
    subprocess.run('apt-get -qq install -y flex bison gawk > /dev/null', shell=True)
RMN = os.path.join(REPO, '.build/chintamani/Romenagri')
if not os.path.exists(os.path.join(RMN, 'rmn2acii')):
    subprocess.run(['bash', os.path.join(REPO, 'scripts/build.sh')], check=True)
os.environ['PATH'] = RMN + os.pathsep + os.environ['PATH']; os.environ['ROMENAGRI'] = RMN
sys.path.insert(0, os.path.join(REPO, 'lang/hindawi/tools'))
import hindawi_chain as hc
GATE = pathlib.Path(REPO, 'samples/gate'); GALLERY = pathlib.Path(REPO, 'docs/hindawi/samples', LANG)

def english_to_language(text, name='program.c'):
    """English source into LANG with BURP; every error is shown, never hidden."""
    d = pathlib.Path(tempfile.mkdtemp(prefix='hindawi-')); (d / 'src').mkdir()
    (d / 'src' / pathlib.Path(name).name).write_text(text, encoding='utf-8')
    r = subprocess.run([sys.executable, os.path.join(REPO, 'lang/hindawi/burp/burp.py'), 'ingest', str(d / 'src'),
                        '--langs', LANG, '--out', str(d / 'n'), '--translators', 'glossary'], capture_output=True, text=True)
    f = d / 'n' / LANG / pathlib.Path(name).name
    if r.returncode != 0 or not f.exists():
        raise RuntimeError('BURP could not put this into ' + LANG + ':\n' + r.stdout + r.stderr)
    return f.read_text(encoding='utf-8')

def program_text(rel):
    """The GATE program rel (a path under samples/gate) as written in LANG: from the gallery, or by BURP."""
    p = GALLERY / rel
    return p.read_text(encoding='utf-8') if p.exists() else english_to_language((GATE / rel).read_text(encoding='utf-8'), rel)

def stages(src, shaili='guru'):
    return hc.stages(src, LANG, 'h2py' if shaili == 'soochee' else 'h2c')

def build(src):
    """Compile a C program written in LANG through the original pipeline: (executable or None, messages)."""
    d = pathlib.Path(tempfile.mkdtemp(prefix='hindawi-')); f = d / 'program.c'; f.write_text(src, encoding='utf-8')
    env = dict(os.environ, HINDAWI_LANG=LANG, HINDAWI_NATIVE_SOURCE='1')
    r = subprocess.run([os.path.join(REPO, 'lang/hindawi/hindawicc'), '-O2', '-o', str(d / 'program'), str(f), '-lm'],
                       capture_output=True, text=True, env=env)
    return (d / 'program' if r.returncode == 0 else None), (r.stderr or 'built')

def run(exe, stdin=''):
    r = subprocess.run([str(exe)], input=stdin, capture_output=True, text=True, timeout=20)
    return r.stdout + r.stderr

def run_python(src, stdin=''):
    r = subprocess.run([sys.executable, '-c', hc.to_py(src, LANG)], input=stdin, capture_output=True, text=True, timeout=20)
    return r.stdout + r.stderr
print('ready:', LANG, 'from', REPO)

## 2. Practice area

Choose **C (guru)** or **Python (soochee)**, a GATE section and a program: it opens in Pashto. Edit it, put any input in the left box, then:

| Button | What it does |
|---|---|
| Build | compiles the C program through the original pipeline |
| Run | runs it with your input (a Python program runs directly) |
| Stages | shows every step: the source, the script filter, the Romenagri form, and the C or Python it becomes |
| To C | shows only the C (or Python) the compiler gets |
| From English | brings the English original of the chosen program into Pashto with BURP |
| New | clears everything, to write your own |
| Upload, Download | open a program from your computer, or save the one in the editor |

In [ ]:
import ipywidgets as w
from IPython.display import display, FileLink
progs = {s: sorted(p.name for p in (GATE / s).iterdir() if p.suffix in ('.c', '.py')) for s in SECTIONS}
shaili = w.ToggleButtons(options=[('C (guru)', 'guru'), ('Python (soochee)', 'soochee')], value='guru')
section = w.Dropdown(options=SECTIONS, description='Section')
program = w.Dropdown(description='Program')
editor = w.Textarea(layout=w.Layout(width='99%', height='340px'))
given = w.Textarea(placeholder='input for the program', layout=w.Layout(width='49%', height='120px'))
shown = w.Textarea(placeholder='output', layout=w.Layout(width='49%', height='120px'))
view = w.Output(layout=w.Layout(width='99%', border='1px solid #bbb', padding='4px'))
upload = w.FileUpload(accept='.c,.py,.txt', multiple=False, description='Upload')
state = {'exe': None}

def fill_programs(*_):
    ext = '.py' if shaili.value == 'soochee' else '.c'
    names = [n for n in progs.get(section.value, []) if n.endswith(ext)]
    program.options = names or ['(none in this section)']
def load(*_):
    state['exe'] = None
    if program.value and not program.value.startswith('('):
        try: editor.value = program_text(section.value + '/' + program.value); shown.value = ''
        except Exception as e: shown.value = str(e)
def do_build(_):
    if shaili.value == 'soochee': shown.value = 'Python needs no build: press Run.'; return
    state['exe'], msg = build(editor.value); shown.value = msg
def do_run(_):
    if shaili.value == 'soochee': shown.value = run_python(editor.value, given.value); return
    shown.value = run(state['exe'], given.value) if state['exe'] else 'Press Build first.'
def do_stages(_):
    view.clear_output()
    with view:
        try:
            for name, text in stages(editor.value, shaili.value):
                print('===== ' + name); print(text)
        except SystemExit as e: print(e)
def do_to_c(_):
    try: shown.value = stages(editor.value, shaili.value)[-1][1]
    except SystemExit as e: shown.value = str(e)
def do_from_english(_):
    rel = section.value + '/' + program.value
    try: editor.value = english_to_language((GATE / rel).read_text(encoding='utf-8'), rel); shown.value = 'from ' + rel
    except Exception as e: shown.value = str(e)
def do_new(_):
    editor.value = given.value = shown.value = ''; state['exe'] = None; view.clear_output()
def do_upload(change):
    v = upload.value
    for f in (v if isinstance(v, (list, tuple)) else v.values()):
        editor.value = bytes(f['content']).decode('utf-8', 'replace')
def do_download(_):
    name = program.value if program.value and not program.value.startswith('(') else 'program.c'
    p = pathlib.Path(name); p.write_text(editor.value, encoding='utf-8')
    if 'google.colab' in sys.modules:
        from google.colab import files; files.download(str(p))
    else:
        with view: display(FileLink(str(p)))

shaili.observe(lambda c: (fill_programs(), load()), 'value')
section.observe(lambda c: (fill_programs(), load()), 'value')
program.observe(lambda c: load(), 'value')
upload.observe(do_upload, 'value')
acts = [('Build', do_build), ('Run', do_run), ('Stages', do_stages), ('To C', do_to_c),
        ('From English', do_from_english), ('New', do_new), ('Download', do_download)]
bar = w.HBox([w.Button(description=n, layout=w.Layout(width='auto')) for n, _ in acts] + [upload])
for b, (_, f) in zip(bar.children, acts): b.on_click(f)
fill_programs(); load()
display(w.VBox([shaili, w.HBox([section, program]), bar, editor, w.HBox([given, shown]), view]))

## 3. Teach GATE in your language

Each section below is a place to explain the topic in your own words, beside its programs. Write in the cell, or claim the section as a work package on the work-packages page of hindawiai/hindawi2026.10, so the explanation joins the repository with your name on it.

### algorithms: analysis

Programs: `growth_rates.c`

*Explain this section in Pashto here.*

### algorithms: backtracking

Programs: `n_queens.c`

*Explain this section in Pashto here.*

### algorithms: divide and conquer

Programs: `maximum_subarray.c`, `towers_of_hanoi.c`

*Explain this section in Pashto here.*

### algorithms: dynamic programming

Programs: `edit_distance.c`, `knapsack_01.c`, `longest_common_subsequence.c`, `matrix_chain.c`

*Explain this section in Pashto here.*

### algorithms: graphs

Programs: `breadth_first_search.c`, `depth_first_search.c`, `dijkstra.c`, `floyd_warshall.c`, `kruskal.c`, `prim.c`, `topological_sort.c`

*Explain this section in Pashto here.*

### algorithms: greedy

Programs: `activity_selection.c`, `fractional_knapsack.c`, `huffman_code_lengths.c`

*Explain this section in Pashto here.*

### algorithms: searching

Programs: `binary_search.c`

*Explain this section in Pashto here.*

### algorithms: sorting

Programs: `heap_sort.c`, `insertion_sort.c`, `merge_sort.c`, `quick_sort.c`

*Explain this section in Pashto here.*

### compiler design

Programs: `activation_records.c`, `common_subexpression.c`, `constant_folding.c`, `infix_to_postfix.c`, `liveness_analysis.c`, `recursive_descent_parser.c`, `rpn_evaluator.c`, `three_address_code.c`, `tokenizer.c`

*Explain this section in Pashto here.*

### computer networks

Programs: `aloha_efficiency.c`, `cidr.c`, `crc32.c`, `delay_throughput.c`, `distance_vector.c`, `dns_name_encoding.c`, `http_request.c`, `internet_checksum.c`, `ip_fragmentation.c`, `link_state.c`, `nat_table.c`, `sliding_window.c`, `socket_pair.c`, `tcp_congestion.c`

*Explain this section in Pashto here.*

### computer organization

Programs: `addressing_modes.c`, `alu_flags.c`, `cache_mapping.c`, `interrupt_dma.c`, `memory_access_time.c`, `microprogrammed_control.c`, `pipeline_hazards.c`

*Explain this section in Pashto here.*

### data structures

Programs: `adjacency_list.c`, `binary_search_tree.c`, `circular_queue.c`, `hash_table.c`, `linked_list.c`, `min_heap.c`, `stack.c`, `tree_traversals.c`

*Explain this section in Pashto here.*

### databases

Programs: `attribute_closure.c`, `bplus_tree_order.c`, `candidate_keys.c`, `conflict_serializability.c`, `hash_join.c`, `relational_algebra.c`

*Explain this section in Pashto here.*

### digital logic

Programs: `floating_point.c`, `gray_code.c`, `quine_mccluskey.c`, `ripple_carry_adder.c`, `sequence_detector.c`, `twos_complement_fixed_point.c`

*Explain this section in Pashto here.*

### discrete mathematics

Programs: `bipartite_matching.c`, `graph_colouring.c`, `group_axioms.c`, `permutations.c`, `recurrence_table.c`, `relation_properties.c`, `sieve_of_eratosthenes.c`, `truth_table.c`

*Explain this section in Pashto here.*

### engineering mathematics

Programs: `bayes_theorem.c`, `binomial_poisson.c`, `distribution_sampling.c`, `finite_differences.c`, `gaussian_elimination.c`, `golden_section.c`, `lu_decomposition.c`, `power_iteration.c`, `simpson_integration.c`, `statistics.c`

*Explain this section in Pashto here.*

### operating systems

Programs: `bankers_algorithm.c`, `cpu_scheduling.c`, `disk_scheduling.c`, `fork_wait.c`, `inode_blocks.c`, `lru_page_replacement.c`, `paging_translation.c`, `pipe_ipc.c`, `producer_consumer.c`, `round_robin.c`, `threads_mutex.c`

*Explain this section in Pashto here.*

### paradigms

Programs: `binary_search.py`

*Explain this section in Pashto here.*

### programming

Programs: `gcd_and_lcm.c`, `pointers_and_arrays.c`, `preprocessor_macros.c`, `recursion_fibonacci.c`, `structures_and_unions.c`

*Explain this section in Pashto here.*

### theory of computation

Programs: `cyk_parsing.c`, `dfa_divisible_by_three.c`, `nfa_simulation.c`, `pda_anbn.c`, `regex_matcher.c`, `turing_machine.c`

*Explain this section in Pashto here.*

## 4. Robots

The same language drives the robots: Arduino UNO first, then ESP32. The sketches, in this language and in English, and how to load them, are in `robot_coding/` beside these notebooks.